# NB-08-03 — Attention and Sequences

Scaled dot-product attention computed by hand, then a masked version, then a cost comparison against
recurrence. Pure NumPy: no model download, CPU only.

In [1]:
import numpy as np
print("numpy", np.__version__)

numpy 2.5.3


## 1. Attention by hand

Three tokens, two dimensions, one head. Every row of the attention matrix must sum to 1.

In [2]:
def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def attention(Q, K, V, mask=None):
    scores = Q @ K.T / np.sqrt(Q.shape[-1])
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    w = softmax(scores)
    return w @ V, w

X = np.array([[1.0, 0.0], [0.9, 0.1], [0.7, 0.7]])
out, w = attention(X, X, X)
print("weights:\n", np.round(w, 3))
print("row sums:", np.round(w.sum(axis=1), 6))
print("output:\n", np.round(out, 3))

weights:
 [[0.365 0.34  0.295]
 [0.355 0.336 0.309]
 [0.311 0.311 0.379]]
row sums: [1. 1. 1.]
output:
 [[0.877 0.241]
 [0.874 0.25 ]
 [0.855 0.296]]


## 2. Why the sqrt(d_k) matters

Without scaling, larger dimensions produce larger logits and a saturated softmax — one token takes
almost all the weight.

In [3]:
d = 64
rng = np.random.default_rng(0)
Q = rng.normal(size=(6, d)); K = rng.normal(size=(6, d))
scaled = softmax(Q @ K.T / np.sqrt(d))
unscaled = softmax(Q @ K.T)
print("scaled   max weight:", round(float(scaled.max()), 3), "min:", round(float(scaled.min()), 4))
print("unscaled max weight:", round(float(unscaled.max()), 3), "min:", round(float(unscaled.min()), 6))

scaled   max weight: 0.647 min: 0.0131
unscaled max weight: 1.0 min: 0.0


## 3. Causal masking

A language model may not look ahead. The mask makes the future impossible, not merely unlikely.

In [4]:
n = 4
causal = np.tril(np.ones((n, n), dtype=bool))
X4 = rng.normal(size=(n, 4))
out4, w4 = attention(X4, X4, X4, mask=causal)
print("mask:\n", causal.astype(int))
print("weights:\n", np.round(w4, 3))
print("last row weights:", np.round(w4[-1], 3), "(only the last position, as intended)")

mask:
 [[1 0 0 0]
 [1 1 0 0]
 [1 1 1 0]
 [1 1 1 1]]
weights:
 [[1.    0.    0.    0.   ]
 [0.036 0.964 0.    0.   ]
 [0.019 0.003 0.978 0.   ]
 [0.015 0.016 0.08  0.889]]
last row weights: [0.015 0.016 0.08  0.889] (only the last position, as intended)


## 4. Cost: attention versus recurrence

Attention is quadratic in sequence length but has no sequential dependency; recurrence is linear in
length but strictly ordered. Count the operations.

In [5]:
for n in (128, 512, 1024):
    for d in (64,):
        attn = n * n * d
        ff = n * d * d
        rnn = n * d * d
        print(f"n={n:5d} d={d}: attention={attn:>10,d}  feed-forward={ff:>10,d}  rnn={rnn:>10,d}")

n=  128 d=64: attention= 1,048,576  feed-forward=   524,288  rnn=   524,288
n=  512 d=64: attention=16,777,216  feed-forward= 2,097,152  rnn= 2,097,152
n= 1024 d=64: attention=67,108,864  feed-forward= 4,194,304  rnn= 4,194,304


## 5. Positional information

Attention alone is permutation-invariant: shuffle the tokens and the outputs shuffle with them. Show it,
then add a positional encoding and show the outputs no longer match.

In [6]:
idx = [0, 2, 1]
perm_out, _ = attention(X[idx], X[idx], X[idx])
print("shuffled-output rows equal unshuffled rows shuffled:",
      np.allclose(perm_out, out[idx]))

def positional(n, d):
    pos = np.arange(n)[:, None]
    i = np.arange(d)[None, :]
    angle = pos / np.power(10000, (2 * (i // 2)) / d)
    enc = np.zeros((n, d))
    enc[:, 0::2] = np.sin(angle[:, 0::2])
    enc[:, 1::2] = np.cos(angle[:, 1::2])
    return enc

Xp = X + positional(3, 2) * 0.5
perm_out_p, _ = attention(Xp[idx], Xp[idx], Xp[idx])
print("with positional encoding, permutation changes the result:",
      not np.allclose(perm_out_p, out[idx]))

shuffled-output rows equal unshuffled rows shuffled: True
with positional encoding, permutation changes the result: True


## What you should take away

- Attention is a weighted average: rows of the weight matrix sum to 1, so it cannot invent content.
- The $\sqrt{d_k}$ scaling keeps the softmax from saturating as dimension grows.
- A causal mask enforces order for language modelling.
- Attention costs $O(n^2d)$; recurrence costs $O(nd^2)$ but cannot be parallelised over time.
- Without positional encodings, order is invisible — verified above, not asserted.